# 03 - Un dataset robuste : 1 ligne par partie, cible = Elo moyen

**But** : construire un jeu de données **fiable et prêt pour le ML** pour prédire `elo_moyen = (EloBlancs + EloNoirs) / 2`
à partir des coups (SAN) et de la pendule.

Méthode : pour chaque point de qualité on **mesure d'abord**, puis on **décide** (seuils justifiés par les données).
Rien n'est filtré avant la section 8 : les sections 3 à 7 sont un audit, la section 8 applique toutes les décisions d'un coup.

| # | Contenu |
|---|---|
| 1-2 | Paramètres, chargement |
| 3 | Audit structure (colonnes, tournois, couverture horaire) |
| 4 | Audit des coups (parsing, annotations parasites, `%eval`, légalité) |
| 5 | Audit de la pendule |
| 6 | Audit de la cible (Elo, écart entre joueurs, Elo provisoires) |
| 7 | Audit des joueurs (concentration, doublons) |
| 8 | Application des filtres |
| 9 | Plafond de parties par joueur + split train/valid/test **par joueur** |
| 10 | Table finale, contrôles automatiques, biais de sélection |
| 11 | Sauvegarde et fiche du dataset |

## 1. Paramètres

In [ ]:
import sys
from pathlib import Path

import chess
import matplotlib.pyplot as plt
import plotly.express as px
import polars as pl

sys.path.append(str(Path("../src/utils").resolve()))
from pgn import read_games

PGN_PATH = Path("../Data/lichess_db_standard_rated_2026-09.pgn.zst")
RAW_PATH = Path("../Data/blitz_180_raw_full.parquet")  # 2M premières parties, cadences 180+0 / 180+2
OUT_PATH = Path("../Data/blitz_180_parties.parquet")

SEED = 42
# Seuils : chacun est justifié dans la section indiquée
MIN_DEMI_COUPS = 20   # section 8
ECART_MAX = 200       # section 6.2 : |EloBlancs - EloNoirs| maximal
RD_MAX = 9            # section 6.3 : |RatingDiff| maximal (sinon Elo provisoire)
MAX_PAR_JOUEUR = 20   # section 7 : plafond de parties par joueur
PART_TRAIN, PART_VALID = 0.7, 0.15  # section 9.2 (le reste = test)

pl.Config.set_tbl_rows(25)
plt.rcParams["figure.dpi"] = 110

## 2. Chargement

Lecture streaming des 2M premières parties du PGN, filtrées sur 180+0 / 180+2 (~1 min la première fois, puis cache parquet).
Pourquoi une seule famille de cadence : l'Elo Lichess est propre à chaque cadence, et la pendule n'est comparable que
si tous partent du même temps.

In [ ]:
if not RAW_PATH.exists():
    read_games(PGN_PATH, n_read=2_000_000, time_controls={"180+0", "180+2"}).write_parquet(RAW_PATH)

raw = pl.read_parquet(RAW_PATH).drop("Movetext")  # tout est déjà dans `moves` : {san, clk, eval}
print(raw.shape, f"{raw.estimated_size('gb'):.1f} Go")
raw.select("White", "WhiteElo", "Black", "BlackElo", "Result", "TimeControl", "Termination", "n_moves").head(3)

On ajoute les colonnes de travail dont l'audit a besoin (aucune ligne n'est retirée) :
- `game_id` : identifiant Lichess de la partie ;
- `san_brut` : coups tels quels ; `clk` : pendule en secondes après chaque demi-coup ;
- `a_eval` : la partie contient au moins un `%eval`.

In [ ]:
coup = pl.element().struct
t = coup.field("clk").str.to_time("%H:%M:%S")
en_secondes = t.dt.hour().cast(pl.Int32) * 3600 + t.dt.minute().cast(pl.Int32) * 60 + t.dt.second().cast(pl.Int32)

games = raw.with_columns(
    game_id=pl.col("Site").str.extract(r"lichess\.org/(\w+)"),
    tournoi=pl.col("Event").str.contains("tournament"),
    san_brut=pl.col("moves").list.eval(coup.field("san")),
    clk=pl.col("moves").list.eval(en_secondes),
    a_eval=pl.col("moves").list.eval(coup.field("eval").is_not_null()).list.any(),
    ecart=(pl.col("WhiteElo") - pl.col("BlackElo")).abs(),
    inc=pl.col("TimeControl").str.extract(r"\+(\d+)").cast(pl.Int32),  # incrément : 0 ou 2 s
).drop("moves")
del raw
games.height

## 3. Audit de la structure

In [ ]:
# Valeurs manquantes + colonnes constantes (qui n'apportent rien)
manquants = games.null_count().transpose(include_header=True, column_names=["n_null"]).filter(pl.col("n_null") > 0)
constantes = [c for c in games.columns if games[c].dtype in (pl.String, pl.Int32, pl.Boolean) and games[c].n_unique() == 1]
print("colonnes constantes :", constantes)
manquants

- `Round`, `UTCDate`, `Date` : constantes → inutiles. `Event` ne sert qu'à repérer les tournois.
- `RatingDiff` manque pour les parties abandonnées (pas de changement d'Elo). Les titres manquent pour les joueurs non titrés (normal).
- La couverture horaire et la part des tournois comptent pour la suite :

In [ ]:
horaire = (
    games.group_by(heure=pl.col("UTCTime").str.slice(0, 2).cast(pl.Int32))
    .agg(parties=pl.len(), elo_moyen=((pl.col("WhiteElo") + pl.col("BlackElo")) / 2).mean().round())
    .sort("heure")
)
print(f"part des tournois (Arena) : {games['tournoi'].mean():.1%}")
horaire.transpose(include_header=True)

- Les 2M parties lues ne couvrent que **00h-16h UTC** (pas 24 h) du 1er septembre : un seul jour, pas toutes les heures.
  L'Elo moyen est plat selon l'heure (≈ 1680) : pas de biais horaire visible. Mais le dataset reste *un jour* de Lichess ;
  pour plus de diversité il faudrait lire d'autres jours du fichier.
- ~9 % des parties sont des **tournois Arena** : leurs appariements sont très différents (section 6.2).

## 4. Audit des coups

### 4.1 Le parsing est-il fiable ?
On compare le nombre de coups extraits (`n_moves`) au nombre de numéros « `12.` / `12...` » du texte brut.
Ici le texte brut est relu depuis le parquet d'origine pour la comparaison.

In [ ]:
brut = pl.read_parquet(RAW_PATH, columns=["Movetext", "n_moves"])
print("parties où n_moves != nombre de numéros de coups :",
      brut.filter(pl.col("Movetext").str.count_matches(r"\d+\.+ ") != pl.col("n_moves")).height)
del brut

### 4.2 Annotations parasites dans les coups
Certains coups finissent par `?`, `?!`, `??`, `!` : ce sont des annotations du moteur d'analyse de Lichess.

In [ ]:
games.with_columns(glyphe=pl.col("san_brut").list.eval(pl.element().str.contains(r"[?!]")).list.any()).group_by("glyphe", "a_eval").agg(
    parties=pl.len()
).sort("glyphe", "a_eval")

**Toutes** les parties annotées sont des parties analysées (`%eval`), et jamais l'inverse. Or l'analyse n'existe que si quelqu'un
l'a demandée (surtout des joueurs forts) : ces glyphes sont un **indice caché sur le niveau**, absent des autres parties.
Un modèle les apprendrait comme raccourci (fuite). Décision : on les retire des coups (`Qxf7?!` → `Qxf7`).

In [ ]:
glyphes = r"[?!]+$"
games = games.with_columns(san=pl.col("san_brut").list.eval(pl.element().str.replace(glyphes, ""))).drop("san_brut")

syntaxe_san = r"^([PNBRQK]?[a-h]?[1-8]?x?[a-h][1-8](=[NBRQ])?|O-O(-O)?)[+#]?$"
invalides = games.select("san").explode("san").filter(~pl.col("san").str.contains(syntaxe_san))
print("coups à la syntaxe invalide après nettoyage :", invalides.height)

### 4.3 `%eval` : à garder ?

In [ ]:
(
    games.with_columns(tranche=((pl.col("WhiteElo") + pl.col("BlackElo")) / 2 // 200 * 200).clip(1000, 2600))
    .group_by("tranche")
    .agg(parties=pl.len(), pct_avec_eval=(pl.col("a_eval").mean() * 100).round(1))
    .sort("tranche")
)

La part de parties évaluées est stable (~8-10 %) jusqu'à 2000 puis **explose** au-dessus (~20 % à 2200, ~50 % à 2600) :
`%eval` (rare, biaisé) est lui aussi un indice sur la cible, pas sur les coups.
Décision : on **n'utilise pas `%eval`** (et pas de Stockfish dans ce projet). Les parties évaluées restent dans le dataset,
une fois les glyphes retirés, pour ne pas biaiser la population ; la colonne `a_eval` n'est pas conservée.

### 4.4 Légalité des parties et cohérence du résultat
On rejoue un échantillon de parties avec `python-chess` : un seul coup illégal = parsing ou donnée corrompue.

In [ ]:
def rejoue(coups):
    plateau = chess.Board()
    for c in coups:
        plateau.push_san(c)  # lève une exception si le coup est illégal / ambigu
    return plateau


def verifie_legalite(table, n=20_000):
    illegales = 0
    for coups in table.sample(min(n, table.height), seed=SEED)["san"].to_list():
        try:
            rejoue(coups)
        except ValueError:
            illegales += 1
    return illegales


print("parties illégales sur 20 000 :", verifie_legalite(games))

In [ ]:
# Un coup final avec # (mat) doit donner la victoire au joueur qui vient de jouer
mats = games.filter(pl.col("san").list.last().str.ends_with("#"))
gagnant_attendu = pl.when(pl.col("n_moves") % 2 == 1).then(pl.lit("1-0")).otherwise(pl.lit("0-1"))
print("parties finies par mat :", mats.height, "| résultat incohérent :", mats.filter(pl.col("Result") != gagnant_attendu).height)

Aucune partie illégale, aucun mat incohérent : les coups sont fiables. Le résultat n'est de toute façon **pas gardé comme variable** :
il n'est pas « contenu dans les coups » et dépend du niveau relatif des deux joueurs, pas du niveau absolu.

## 5. Audit de la pendule

La pendule (`clk`) donne le temps restant après chaque demi-coup. Blancs = indices pairs, Noirs = indices impairs.
Trois choses peuvent la rendre non comparable : absente, départ ≠ 180 s (**berserk** en tournoi : temps divisé par 2),
ou **temps ajouté** par l'adversaire (la pendule remonte de plus que l'incrément).

In [ ]:
def clk_joueur(offset):  # 0 = Blancs, 1 = Noirs
    return pl.col("clk").list.gather_every(2, offset)


def depart(offset):
    return clk_joueur(offset).list.first()


def remontee(offset):  # plus forte remontée de la pendule entre deux coups du joueur
    return clk_joueur(offset).list.diff().list.max()


games.select(
    sans_pendule=pl.col("clk").list.eval(pl.element().is_null()).list.any().sum(),
    berserk=((depart(0) < 180) | (depart(1) < 180)).sum(),
    depart_inattendu_haut=((depart(0) > 182) | (depart(1) > 182)).sum(),
    temps_ajoute=((remontee(0) > pl.col("inc")) | (remontee(1) > pl.col("inc"))).sum(),
    pendule_negative=(pl.col("clk").list.min() < 0).sum(),
)

In [ ]:
# Le berserk est-il surtout un phénomène de tournoi ?
games.group_by("tournoi").agg(
    parties=pl.len(),
    pct_berserk=(((depart(0) < 180) | (depart(1) < 180)).mean() * 100).round(2),
)

- Le berserk est un phénomène de **tournoi** (~26 % des parties d'Arena contre 0,04 % des parties libres).
- Départ : presque toujours 180 s (normal) ou 90 s (berserk) ; les quelques départs > 182 s sont aussi écartés.
- Aucune pendule négative. Les parties vides (abandonnées avant le 1er coup) ont une liste vide : elles tombent avec le filtre de longueur.
- Décision : on retire les parties avec berserk ou temps ajouté (pour les deux joueurs), afin que toutes les pendules soient comparables.

## 6. Audit de la cible

### 6.1 Les Elo sont-ils cohérents ?
Lichess donne l'Elo **avant** la partie et `RatingDiff` = sa variation. Donc, pour un joueur qui enchaîne deux parties,
`Elo(partie suivante) = Elo(partie précédente) + RatingDiff(partie précédente)`. Si c'est vrai, les Elo sont fiables.

In [ ]:
cote = lambda c: games.select("UTCTime", joueur=pl.col(c), elo=pl.col(f"{c}Elo"), rd=pl.col(f"{c}RatingDiff"))
chaine = (
    pl.concat([cote("White"), cote("Black")])
    .sort("joueur", "UTCTime")
    .with_columns(attendu=(pl.col("elo") + pl.col("rd")).shift(1).over("joueur"))
    .drop_nulls("attendu")
    .with_columns(erreur=(pl.col("elo") - pl.col("attendu")).abs())
)
print(f"{chaine.height} paires de parties consécutives | Elo exactement conforme : {(chaine['erreur'] == 0).mean():.1%}")
del chaine

Les Elo sont **cohérents à ~99 %** (les écarts viennent probablement de parties jouées dans une autre cadence entre-temps, hors de notre extrait).
Les Elo ne sont donc pas bruités par le format du fichier. Reste le problème des Elo *incertains* (6.3) et des écarts (6.2).

### 6.2 Écart d'Elo entre les deux joueurs
La cible est une **moyenne** : elle n'a de sens que si les deux joueurs ont un niveau proche. Si l'écart est de 800 points,
`elo_moyen` ne décrit ni l'un ni l'autre. L'erreur maximale en prédisant la moyenne pour un des deux joueurs est `écart / 2`.

In [ ]:
games.group_by("tournoi").agg(
    parties=pl.len(),
    ecart_median=pl.col("ecart").median(),
    ecart_p90=pl.col("ecart").quantile(0.9),
    ecart_p99=pl.col("ecart").quantile(0.99),
    ecart_max=pl.col("ecart").max(),
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(games["ecart"].clip(upper_bound=800), bins=80, log=True)
ax.axvline(ECART_MAX, color="red", ls="--", label=f"seuil retenu : {ECART_MAX}")
ax.set(xlabel="|EloBlancs - EloNoirs| (plafonné à 800)", ylabel="parties (log)", title="Écart d'Elo entre les deux joueurs")
ax.legend()
plt.show()

pl.DataFrame(
    {"ecart_max": s, "parties_gardées_%": round((games["ecart"] <= s).mean() * 100, 1), "erreur_max_en_predisant_la_moyenne": s / 2}
    for s in [50, 100, 150, 200, 300, 500]
)

- En parties libres l'écart médian est ≈ 30 points (99 % sous ~185) ; en **tournoi** la médiane est ≈ 5× plus grande et 1 % des parties dépasse ~880.
- Seuil retenu : **écart ≤ 200** (garde ~96 %). L'« erreur de cible » due à la moyenne reste ≤ 100 points, soit ~1/3 du
  baseline (MAE ≈ 281) : acceptable. Au-delà, on perd vite en pertinence de la cible pour très peu de lignes gagnées.
- Conséquence : la plupart des tournois disparaîtront aussi (part restante affichée en section 8).

### 6.3 Les Elo provisoires
Un compte neuf démarre à 1500 avec une incertitude forte (son Elo bouge beaucoup à chaque partie). `RatingDiff` mesure cette
variation. On regarde, à écart ≤ 200 (pour que l'écart d'Elo n'explique pas à lui seul un gros `RatingDiff`), comment l'Elo moyen
évolue selon |RatingDiff| : s'il converge vers 1500, ce sont des comptes neufs.

In [ ]:
par_rd = (
    pl.concat([
        games.filter(pl.col("ecart") <= ECART_MAX).select(elo=pl.col("WhiteElo"), rd=pl.col("WhiteRatingDiff")),
        games.filter(pl.col("ecart") <= ECART_MAX).select(elo=pl.col("BlackElo"), rd=pl.col("BlackRatingDiff")),
    ])
    .drop_nulls()
    .group_by(rd_abs=pl.col("rd").abs().clip(0, 25))
    .agg(joueurs_parties=pl.len(), elo_moyen=pl.col("elo").mean())
    .sort("rd_abs")
)
fig = px.line(par_rd, x="rd_abs", y="elo_moyen", markers=True, title="Elo moyen selon |RatingDiff| (25 = 25 et plus)")
fig.add_vline(x=RD_MAX + 0.5, line_dash="dash", line_color="red")
fig.show()
par_rd.filter(pl.col("rd_abs").is_between(4, 14)).with_columns(pl.col("elo_moyen").round())

- La grande masse des variations est entre 4 et 8 points, avec un Elo moyen ≈ 1660-1700.
- À partir de |RatingDiff| ≈ 10, l'Elo moyen **chute vers 1500** (valeur de départ d'un compte neuf) : ce sont des Elo encore peu fiables.
- Seuil retenu : **|RatingDiff| ≤ 9 pour les deux joueurs** (il suffit d'un joueur provisoire pour fausser la moyenne).
  Les parties sans `RatingDiff` (abandons) sont retirées de fait.

## 7. Audit des joueurs et des doublons

### 7.1 Concentration : des joueurs très actifs
Un jour de données contient des joueurs qui font des centaines de parties. S'ils pèsent trop, le modèle apprend *ces joueurs*
plutôt que « le niveau ».

In [ ]:
parties_par_joueur = pl.concat([games["White"], games["Black"]]).value_counts().rename({"White": "joueur", "count": "n"})
print("joueurs distincts :", parties_par_joueur.height, "| places :", 2 * games.height)
print(parties_par_joueur["n"].quantile(0.5), "partie(s) pour le joueur médian ; max :", parties_par_joueur["n"].max())
part = lambda k: parties_par_joueur.filter(pl.col("n") >= k)["n"].sum() / (2 * games.height)
pl.DataFrame({"joueurs_avec_au_moins_n_parties": [5, 10, 20, 50], "part_des_lignes_%": [round(part(k) * 100, 1) for k in [5, 10, 20, 50]]})

Un tiers des lignes vient de joueurs à ≥ 20 parties. Décision (section 9) : **plafonner à 20 parties par joueur**
(tirage aléatoire reproductible). On perd ~20 % des parties mais aucun joueur ne domine.

### 7.2 Mêmes adversaires en boucle, parties identiques

In [ ]:
paires = games.group_by(paire=pl.concat_list(pl.min_horizontal("White", "Black"), pl.max_horizontal("White", "Black"))).len().sort("len", descending=True)
print("paires qui jouent ≥ 5 fois ensemble :", paires.filter(pl.col("len") >= 5).height, "| record :", paires["len"].max())

sequence = games.filter(pl.col("n_moves") >= MIN_DEMI_COUPS).with_columns(suite=pl.col("san").list.join(" "))
print("parties longues (≥ 20 demi-coups) dont toute la suite de coups est un doublon :", sequence.height - sequence["suite"].n_unique())
sequence.group_by("suite").len().filter(pl.col("len") > 1).sort("len", descending=True).head(3).select("len", pl.col("suite").str.slice(0, 70))

- Des paires jouent 100+ fois ensemble (revanches en boucle, parfois du farming d'Elo) : le plafond par joueur les limite.
- Parmi les parties longues, quelques **doublons exacts** (scripts / comptes automatiques non marqués BOT) : on garde la première occurrence.
- Les ouvertures de théorie jouées à l'identique sur 20 demi-coups sont, elles, légitimes : on les garde.

## 8. Application des filtres

Tous les filtres au même endroit, chacun = condition pour **garder** la partie. Le tableau montre ce que chaque filtre retire.

In [ ]:
def pendule_ok(offset):
    return (depart(offset) == 180) & (remontee(offset) <= pl.col("inc"))


est_bot = (pl.col("WhiteTitle") == "BOT").fill_null(False) | (pl.col("BlackTitle") == "BOT").fill_null(False)

filtres = {
    "BOT": ~est_bot,
    "Abandoned / Rules infraction": ~pl.col("Termination").is_in(["Abandoned", "Rules infraction"]),
    f"moins de {MIN_DEMI_COUPS} demi-coups": pl.col("n_moves") >= MIN_DEMI_COUPS,
    "pendule manquante": ~pl.col("clk").list.eval(pl.element().is_null()).list.any(),
    "pendule anormale (berserk, temps ajouté)": pendule_ok(0) & pendule_ok(1),
    f"écart d'Elo > {ECART_MAX}": pl.col("ecart") <= ECART_MAX,
    f"Elo provisoire (|RatingDiff| > {RD_MAX})": (pl.col("WhiteRatingDiff").abs() <= RD_MAX) & (pl.col("BlackRatingDiff").abs() <= RD_MAX),
}

clean, bilan = games, []
for nom, garder in filtres.items():
    avant = clean.height
    clean = clean.filter(garder)  # null -> retiré
    bilan.append({"filtre": nom, "retirées": avant - clean.height, "restantes": clean.height})

avant = clean.height
clean = clean.unique("san", keep="first", maintain_order=True)  # doublons exacts de toute la suite de coups
bilan.append({"filtre": "doublon exact de la partie", "retirées": avant - clean.height, "restantes": clean.height})
print(f"part des tournois : {games['tournoi'].mean():.1%} au départ -> {clean['tournoi'].mean():.1%} après filtres")
pl.DataFrame(bilan).with_columns((pl.col("retirées") / games.height * 100).round(2).alias("%_du_départ"))

## 9. Plafond par joueur et split par joueur

### 9.1 Plafond de parties par joueur
Pour chaque joueur on tire un rang aléatoire sur ses parties ; une partie est gardée si **ses deux joueurs** sont dans leurs
`MAX_PAR_JOUEUR` premières parties.

In [ ]:
cotes = lambda c: clean.select("game_id", joueur=pl.col(c), alea=pl.int_range(pl.len()).shuffle(seed=SEED))
rangs = (
    pl.concat([cotes("White"), cotes("Black")])
    .with_columns(rang=pl.col("alea").rank("ordinal").over("joueur"))
    .group_by("game_id")
    .agg(rang_max=pl.col("rang").max())
)
avant = clean.height
clean = clean.join(rangs.filter(pl.col("rang_max") <= MAX_PAR_JOUEUR), on="game_id")
print(f"plafond {MAX_PAR_JOUEUR} parties/joueur : {avant} -> {clean.height} parties")

### 9.2 Split train / valid / test **par joueur**
Un même joueur garde à peu près le même Elo toute la journée (écart-type de quelques points) et un style reconnaissable.
Si ses parties sont à la fois en train et en test, le modèle peut *reconnaître le joueur* : le score test est trop beau.

Chaque partie ayant 2 joueurs, on attribue un fold à chaque **joueur** (aléatoire, reproductible) et on ne garde que les parties
dont les **deux joueurs sont dans le même fold**. Les parties « croisées » sont écartées : les mettre dans un fold fuirait vers l'autre.

In [ ]:
joueurs = pl.concat([clean["White"], clean["Black"]]).unique().sort()
fold = (
    pl.DataFrame({"joueur": joueurs})
    .with_columns(u=pl.int_range(pl.len()).shuffle(seed=SEED) / pl.len())
    .with_columns(
        split=pl.when(pl.col("u") < PART_TRAIN).then(pl.lit("train"))
        .when(pl.col("u") < PART_TRAIN + PART_VALID).then(pl.lit("valid"))
        .otherwise(pl.lit("test"))
    )
    .select("joueur", "split")
)

avant = clean.height
clean = (
    clean.join(fold.rename({"joueur": "White", "split": "split_blanc"}), on="White")
    .join(fold.rename({"joueur": "Black", "split": "split_noir"}), on="Black")
)
print("parties croisées (joueurs dans 2 folds différents) écartées :", clean.filter(pl.col("split_blanc") != pl.col("split_noir")).height, "sur", avant)
clean = clean.filter(pl.col("split_blanc") == pl.col("split_noir")).rename({"split_blanc": "split"}).drop("split_noir")
clean.group_by("split").agg(parties=pl.len()).with_columns(pct=(pl.col("parties") / pl.col("parties").sum() * 100).round(1)).sort("parties", descending=True)

Le compromis : avec 70/15/15 par joueur, on garde ≈ 53 % des parties (0,7² + 0,15² + 0,15²) et la part de chaque split en parties est
très différente de la part en joueurs (le train pèse ~90 % des parties, valid et test ~5 % chacun). Un split 80/10/10 ne laissait
que ~3 700 parties en test (erreur d'estimation de la MAE trop grande) ; 70/15/15 en donne ~2× plus.
Pour plus de stabilité on pourra aussi faire une validation croisée **par groupes de joueurs** sur le train.

## 10. Table finale et contrôles

Colonnes gardées : la **cible** `elo_moyen`, les **coups** `san` et la **pendule** `clk` (même indexation : Blancs = indices pairs),
`cadence`, `eco`, `n_moves`, et pour le split : `blanc`, `noir`, `split`.
Retirés volontairement : Elo individuels et `RatingDiff` (fuite : Elo adverse ≈ Elo du joueur, corrélation 0,96),
résultat et fin de partie, `%eval`, tournoi.

In [ ]:
parties = clean.select(
    "game_id",
    "split",
    blanc=pl.col("White"),
    noir=pl.col("Black"),
    elo_moyen=(pl.col("WhiteElo") + pl.col("BlackElo")) / 2,
    cadence=pl.col("TimeControl"),
    eco=pl.col("ECO"),
    n_moves=pl.col("n_moves"),
    san=pl.col("san"),
    clk=pl.col("clk"),
).sort("game_id")
print(parties.shape)
parties.head(3)

In [ ]:
# Contrôles automatiques : si l'un échoue, le dataset n'est pas bon
inc_ = pl.col("cadence").str.extract(r"\+(\d+)").cast(pl.Int32)
joueurs_par_split = pl.concat([parties.select("split", joueur="blanc"), parties.select("split", joueur="noir")]).unique().group_by("joueur").len()

controles = {
    "game_id unique": parties["game_id"].n_unique() == parties.height,
    "aucune valeur manquante": parties.null_count().to_numpy().sum() == 0,
    "elo_moyen entre 400 et 3200": parties["elo_moyen"].min() >= 400 and parties["elo_moyen"].max() <= 3200,
    "len(san) == len(clk) == n_moves": parties.filter((pl.col("san").list.len() != pl.col("n_moves")) | (pl.col("clk").list.len() != pl.col("n_moves"))).height == 0,
    f"n_moves >= {MIN_DEMI_COUPS}": parties["n_moves"].min() >= MIN_DEMI_COUPS,
    "aucune annotation ?! dans les coups": parties.select(pl.col("san").list.eval(pl.element().str.contains(r"[?!]")).list.any().any()).item() is False,
    "syntaxe SAN valide": parties.select("san").explode("san").filter(~pl.col("san").str.contains(syntaxe_san)).height == 0,
    "départ pendule = 180 s (2 joueurs)": parties.filter((depart(0) != 180) | (depart(1) != 180)).height == 0,
    "pendule sans remontée > incrément": parties.filter((remontee(0) > inc_) | (remontee(1) > inc_)).height == 0,
    "aucun joueur dans 2 splits": joueurs_par_split.filter(pl.col("len") > 1).height == 0,
    f"aucun joueur > {MAX_PAR_JOUEUR} parties": pl.concat([parties["blanc"], parties["noir"]]).value_counts()["count"].max() <= MAX_PAR_JOUEUR,
    "parties légales (20 000 rejouées)": verifie_legalite(parties) == 0,
}
for nom, ok in controles.items():
    print("OK " if ok else "ECHEC", nom)
assert all(controles.values())

### Biais de sélection : le dataset ressemble-t-il aux données de départ ?
Filtrer change la population. On compare la cible avant / après, puis entre les trois splits (qui doivent avoir la même distribution).

In [ ]:
quantiles = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
avant = games.select(elo_moyen=(pl.col("WhiteElo") + pl.col("BlackElo")) / 2).with_columns(groupe=pl.lit("départ (toutes les parties)"))
apres = parties.select("elo_moyen").with_columns(groupe=pl.lit("final"))
par_split = parties.select("elo_moyen", groupe="split")

pl.concat([avant, apres, par_split]).group_by("groupe", maintain_order=True).agg(
    parties=pl.len(),
    moyenne=pl.col("elo_moyen").mean().round(),
    ecart_type=pl.col("elo_moyen").std().round(),
    **{f"q{int(q * 100)}": pl.col("elo_moyen").quantile(q).round() for q in quantiles},
)

In [ ]:
histo = (
    parties.group_by("split", elo=pl.col("elo_moyen") // 50 * 50)
    .agg(n=pl.len())
    .with_columns(pct=pl.col("n") / pl.col("n").sum().over("split") * 100)
    .sort("elo")
)
px.line(histo, x="elo", y="pct", color="split", title="Elo moyen par split (% par tranche de 50)").show()

In [ ]:
# Baseline : prédire la médiane du train partout ; on mesure l'erreur sur chaque split
mediane = parties.filter(pl.col("split") == "train")["elo_moyen"].median()
parties.group_by("split").agg(mae_baseline=(pl.col("elo_moyen") - mediane).abs().mean().round(1)).sort("split").with_columns(mediane_train=mediane)

Lecture : si la distribution et la MAE baseline sont proches entre train / valid / test, le split est équilibré.
Les écarts avant / après viennent surtout des Elo provisoires (comptes à 1500) et des tournois retirés ; c'est voulu
(la cible doit refléter un niveau *stable*) mais à garder en tête : le modèle sera moins fiable sur ces profils.

## 11. Sauvegarde et fiche du dataset

In [ ]:
parties.write_parquet(OUT_PATH)
print(OUT_PATH, "->", parties.height, "parties")
parties.schema

### Fiche du dataset `blitz_180_parties.parquet`

| Colonne | Type | Rôle |
|---|---|---|
| `game_id` | str | identifiant Lichess (clé) |
| `blanc`, `noir` | str | pseudos : **uniquement pour grouper/splitter, jamais en feature** |
| `split` | str | `train` / `valid` / `test` (par joueur) |
| `elo_moyen` | float | **cible** |
| `cadence` | str | `180+0` ou `180+2` (feature autorisée) |
| `eco` | str | code d'ouverture (déduit des coups) |
| `n_moves` | int | nombre de demi-coups |
| `san` | list[str] | coups, Blancs aux indices pairs, annotations retirées |
| `clk` | list[int] | pendule restante en secondes après chaque demi-coup (même indexation) |

**Garanties vérifiées** (section 10) : pas de doublon, aucun manquant, coups légaux et sans annotation moteur, pendules comparables,
aucun joueur dans deux splits, ≤ 20 parties par joueur.

**Limites à connaître**
- Un seul jour (1er sept. 2026, 00h-16h UTC), blitz 3 min uniquement : la généralisation à d'autres cadences ou jours n'est pas démontrée.
- Population filtrée : plus de comptes neufs (Elo provisoires), plus de gros écarts d'Elo / tournois.
- Elo proche de la moyenne : les extrêmes (< 1000, > 2400) sont rares ; un modèle les ramènera vers le centre.
- Valid et test sont petits (~5 % des parties chacun) à cause du split par joueur (voir 9.2).

**Étapes suivantes** : features sur `san` / `clk` pour chaque couleur (`gather_every(2, 0)` / `gather_every(2, 1)`),
régression linéaire (à battre : baseline de la section 10), puis n-grammes et modèles séquentiels.